# Taller 4 - Experimentación con MLflow (Palmer Penguins)

Flujo del notebook:

1. Leer los datos **crudos** desde `postgres-data` (`raw.penguins`, cargada por `postgres-init/01-raw.sql`).
2. Preprocesar y guardar los datos **procesados** en `postgres-data` (`processed.penguins`).
3. Leer los procesados **desde la base de datos** y separar train/test.
4. **(TODO)** Configurar MLflow y correr **≥ 20 ejecuciones** variando hiperparámetros.
5. **(TODO)** Registrar el mejor modelo en el Model Registry y asignarle el alias que usa la API.

Las secciones 1-3 reutilizan lo hecho en taller002 (features de pingüinos) y taller003 (raw → clean en BD).
Las secciones marcadas **TODO** son lo nuevo del taller: ver `NOTAS_PENDIENTES.md`.

## 0. Configuración

In [1]:
import os

import mlflow
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split

# Inyectadas por docker-compose.yaml (servicio jupyter).
DATA_DB_URI = os.environ["DATA_DB_URI"]
MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]
MODEL_NAME = os.environ.get("MODEL_NAME", "penguins-classifier")
MODEL_ALIAS = os.environ.get("MODEL_ALIAS", "champion")
EXPERIMENT_NAME = "penguinsTaller004"

engine = create_engine(DATA_DB_URI)

print(f"MLflow:  {MLFLOW_TRACKING_URI}")
print(f"Modelo a registrar: {MODEL_NAME}@{MODEL_ALIAS}")

MLflow:  http://mlflow:5000
Modelo a registrar: penguins-classifier@champion


## 1. Datos crudos (desde la base de datos)

`raw.penguins` se llena una sola vez al crear el volumen de `postgres-data`, con el CSV de `data/penguins.csv`
**sin limpiar** (los `NA` quedan como `NULL`).

In [2]:
df_raw = pd.read_sql("SELECT * FROM raw.penguins ORDER BY rowid", con=engine)
print(df_raw.shape)
print(df_raw.isna().sum())
df_raw.head()

(344, 9)
rowid                 0
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64


,rowid,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,1,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,2,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,3,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,4,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,5,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## 2. Preprocesamiento → `processed.penguins`

Misma limpieza que taller003 / taller002:

- Se quedan las columnas que usa el modelo (`FEATURE_COLUMNS`) + el target (`species`).
- Se eliminan filas con faltantes en esas columnas.
- `rowid` se conserva para poder rastrear cada fila procesada hasta su fila cruda.

Las categóricas (`island`, `sex`) se guardan **como texto**: el encoding queda dentro del
pipeline de sklearn que se registra en MLflow (ver `NOTAS_PENDIENTES.md`, punto 1), así la API
le pasa al modelo los mismos campos que recibe y no tiene que replicar el preprocesamiento.

`FEATURE_COLUMNS` debe coincidir con `api/app/schemas.py`.

In [3]:
FEATURE_COLUMNS = ["island", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]
NUMERIC_COLUMNS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAL_COLUMNS = ["island", "sex"]
TARGET = "species"

df_processed = (
    df_raw[["rowid", *FEATURE_COLUMNS, TARGET]]
    .dropna(subset=[*FEATURE_COLUMNS, TARGET])
    .reset_index(drop=True)
)
df_processed[NUMERIC_COLUMNS] = df_processed[NUMERIC_COLUMNS].astype(float)

# replace: re-ejecutar el notebook deja la tabla consistente con la limpieza actual.
df_processed.to_sql("penguins", con=engine, schema="processed", if_exists="replace", index=False)

print(f"Crudos: {len(df_raw)} filas -> procesados: {len(df_processed)} filas")
df_processed.head()

Crudos: 344 filas -> procesados: 333 filas


,rowid,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,species
0,1,Torgersen,39.1,18.7,181.0,3750.0,male,Adelie
1,2,Torgersen,39.5,17.4,186.0,3800.0,female,Adelie
2,3,Torgersen,40.3,18.0,195.0,3250.0,female,Adelie
3,5,Torgersen,36.7,19.3,193.0,3450.0,female,Adelie
4,6,Torgersen,39.3,20.6,190.0,3650.0,male,Adelie


## 3. Leer los procesados desde la base de datos y separar train/test

In [4]:
df = pd.read_sql("SELECT * FROM processed.penguins ORDER BY rowid", con=engine)

X = df[FEATURE_COLUMNS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(y.value_counts())

Train: (266, 6), Test: (67, 6)
species
Adelie       146
Gentoo       119
Chinstrap     68
Name: count, dtype: int64


## 4. TODO - Configurar MLflow

Ver `NOTAS_PENDIENTES.md`, punto 1.

- Apuntar el cliente al servidor (`MLFLOW_TRACKING_URI`).
- Crear/seleccionar el experimento.

In [5]:
# TODO(taller004): configurar tracking URI y experimento.

In [6]:
mlflow.set_tracking_uri

<function mlflow.tracking._tracking_service.utils.set_tracking_uri(uri: str | pathlib.Path) -> None>

In [14]:
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1791051439227, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791051439227, lifecycle_stage='active', name='penguinsTaller004', tags={}, trace_location=None, workspace='default'>

In [8]:
MLFLOW_TRACKING_URI,

('http://mlflow:5000',)

## 5. TODO - Función de entrenamiento + registro de un run

Ver `NOTAS_PENDIENTES.md`, punto 1. Cada llamada debe ser **un run** de MLflow con:
parámetros, métricas sobre `X_test`, y el modelo (pipeline con el preprocesamiento) con su
firma (`signature`) e `input_example`.

In [ ]:
# TODO(taller004): def train_and_log(...): ...

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Qué hacer con cada grupo de columnas.
#    Cada entrada de la lista es: (nombre_que_tu_eliges, transformador, columnas)
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),   # ¿qué lista de columnas?
    ("num", "passthrough", NUMERIC_COLUMNS),                            # ¿y aquí?
])

# 2. Encadenar preprocesamiento + modelo.
#    Cada paso es: (nombre, objeto)
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
])

# 3. Entrenar: UN solo fit para todo el pipeline
pipeline.fit(X_train, y_train)

# 4. Predecir sobre los datos que el modelo NO vio
y_pred = pipeline.predict(X_test)

# 5. Métricas: (valor_real, valor_predicho)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=1.0000  f1_macro=1.0000


In [30]:
import mlflow
from sklearn.model_selection import StratifiedKFold, cross_val_score
from mlflow.models import infer_signature

def train_and_log(params, run_name):
    with mlflow.start_run(run_name=run_name) as run:          # opcional: un nombre legible, ej. "rf-prueba"
        pipeline = Pipeline([
            ("preprocess", preprocess),
            ("model", RandomForestClassifier(**params, random_state=42, n_jobs=-1)),
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
    
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro"),
        }    
    
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        cv_scores = cross_val_score(
            pipeline,                     # ¿qué se valida? (el objeto completo, sin entrenar o entrenado, da igual: lo clona)
            X_train, y_train,                # ¿con qué datos? (pista: NO los de test)
            cv=cv,
            scoring="f1_macro",
            n_jobs=-1
        )
        
        metrics["cv_f1_macro_mean"] = cv_scores.mean()
        metrics["cv_f1_macro_std"] = cv_scores.std()
    
    
        mlflow.log_params(params)     # registrar parámetros
        mlflow.log_metrics(metrics)    # registrar métricas
    
        signature = infer_signature(X_train, pipeline.predict(X_train))   # (ejemplo de entrada, ejemplo de salida)
    
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            signature=signature,
            input_example=X_train.head(3),
            skops_trusted_types=["sklearn.tree._tree.Tree"],   # ← ¿qué tipo exacto? (está en el error)
        )
        return run.info.run_id, metrics

run_id, metrics = train_and_log(params={"n_estimators": 50, "max_depth": 3}, run_name="rf-prueba2")
print(run_id, metrics)

2026/10/03 19:21:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-prueba2 at: http://mlflow:5000/#/experiments/1/runs/f15337e76ef34f648a868895f4ad2771
🧪 View experiment at: http://mlflow:5000/#/experiments/1
f15337e76ef34f648a868895f4ad2771 {'accuracy': 1.0, 'f1_macro': 1.0, 'cv_f1_macro_mean': np.float64(0.9716708183833443), 'cv_f1_macro_std': np.float64(0.024046132677754767)}


## 6. TODO - Experimentación: al menos 20 ejecuciones

Ver `NOTAS_PENDIENTES.md`, punto 1. Definir la grilla/lista de hiperparámetros (≥ 20 combinaciones)
y llamar a `train_and_log` para cada una.

In [ ]:
# TODO(taller004): grilla de hiperparametros + loop de ejecuciones.

In [31]:
from itertools import product

N_ESTIMATORS = [1, 5, 20, 100, 300]     # de "casi nada" a "mucho", con saltos crecientes
MAX_DEPTH = [1, 2, 4, 8, None]          # None = sin límite de profundidad

grid = [
    {"n_estimators": n, "max_depth": d}   # ← n y d: los valores de ESTA vuelta
    for n, d in product(N_ESTIMATORS, MAX_DEPTH)
]
print(len(grid))   # 25
grid[:3]

25


[{'n_estimators': 1, 'max_depth': 1},
 {'n_estimators': 1, 'max_depth': 2},
 {'n_estimators': 1, 'max_depth': 4}]

In [32]:
results = []
for params in grid:
    run_name = f"rf-n{params['n_estimators']}-d{params['max_depth']}"   # ← único por run
    run_id, metrics = train_and_log(params=params, run_name=run_name)
    results.append({"run_id": run_id, **params, **metrics})
    print(f"{run_name:<14} cv_f1={metrics['cv_f1_macro_mean']:.4f}  test_f1={metrics['f1_macro']:.4f}")
    
df_results = pd.DataFrame(results).sort_values(
    ["cv_f1_macro_mean", "cv_f1_macro_std"],   # primero mejor media; si empatan, menor desviación
    ascending=[False, True],
)
df_results

2026/10/03 19:22:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n1-d1 at: http://mlflow:5000/#/experiments/1/runs/9070c995369645eca90dfc4ab6f0f4d8
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n1-d1       cv_f1=0.5038  test_f1=0.4679


2026/10/03 19:22:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n1-d2 at: http://mlflow:5000/#/experiments/1/runs/3301fca0a90b4d01960cc0da506039f1
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n1-d2       cv_f1=0.9152  test_f1=0.9271


2026/10/03 19:22:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

2026/10/03 19:25:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n300-d8 at: http://mlflow:5000/#/experiments/1/runs/8a322f98d348464e988fe638da2ea0be
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n300-d8     cv_f1=0.9909  test_f1=1.0000


2026/10/03 19:25:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run rf-n300-dNone at: http://mlflow:5000/#/experiments/1/runs/c82d4ef26b2647cbbbd568c54ec00a75
🧪 View experiment at: http://mlflow:5000/#/experiments/1
rf-n300-dNone  cv_f1=0.9909  test_f1=1.0000


,run_id,n_estimators,max_depth,accuracy,f1_macro,cv_f1_macro_mean,cv_f1_macro_std
18,5dcceb47d56040729b85a70be1315b4d,100,8.0,1.000000,1.000000,0.990872,0.011180
19,9d81f963a45d47d899bff1cc4d0abc13,100,NaN,1.000000,1.000000,0.990872,0.011180
23,8a322f98d348464e988fe638da2ea0be,300,8.0,1.000000,1.000000,0.990872,0.011180
24,c82d4ef26b2647cbbbd568c54ec00a75,300,NaN,1.000000,1.000000,0.990872,0.011180
8,784f52d982094d1a92d57649286c2ceb,5,8.0,0.910448,0.908126,0.986705,0.010859
9,a72ce256ace54388ae0e2d0c3a2378ca,5,NaN,0.910448,0.908126,0.983301,0.008734
13,19246b5fe2b2422c8942f68a14a3efdf,20,8.0,0.985075,0.987258,0.983301,0.008734
14,100a0ae328f14d57aaa606dd21c23c61,20,NaN,0.985075,0.987258,0.983301,0.008734
12,39aa84bff08a43f5aa5d0fd97b4d0230,20,4.0,0.985075,0.987258,0.978722,0.014415
7,3b6d244b851540ce8cc26d5213260755,5,4.0,0.970149,0.974396,0.973313,0.008130


## 7. TODO - Elegir el mejor run y registrarlo en el Model Registry

Ver `NOTAS_PENDIENTES.md`, punto 2. El alias asignado debe ser `MODEL_ALIAS`: es el que lee la API.

In [ ]:
# TODO(taller004): buscar el mejor run, registrar el modelo y asignar el alias.

In [ ]:
best = mlflow.search_runs(
    experiment_names=[___],                    # tu experimento
    order_by=["metrics.___ DESC", "metrics.___ ASC"],
    max_results=1,
)
best_run_id = best.iloc[0]["run_id"]
best[["run_id", "tags.mlflow.runName", "params.n_estimators", "params.max_depth", "metrics.cv_f1_macro_mean"]]


## 8. TODO - Verificar: cargar el modelo desde MLflow igual que lo hará la API

Ver `NOTAS_PENDIENTES.md`, punto 2.

In [ ]:
# TODO(taller004): cargar models:/<MODEL_NAME>@<MODEL_ALIAS> y predecir sobre X_test.head().